## PlantGuard‑Meta (OpenEnv) — Kaggle‑compatible Environment Rollouts

This notebook connects to the deployed OpenEnv environment on Hugging Face Spaces and produces:
- `episode_log.csv` (episode-level metrics)
- `reward_curve.png`
- `success_rate.png`

**Deployed Space**: `https://huggingface.co/spaces/Narendra78/plant_governor_env`

### Kaggle notes
- If Kaggle already has the dependencies, you can skip the install cell.
- Outputs are written to the current working directory (Kaggle “working” folder).


In [ ]:
# --- Install dependencies (Kaggle) ---
# If your Kaggle notebook already has these packages, you can skip this cell.

import sys, platform
print('python', sys.version)
print('platform', platform.platform())

# Kaggle supports pip installs, but internet may be restricted depending on settings.
# Uncomment if needed:
# %pip -q install -U "openenv-core[core]>=0.2.2" "pydantic>=2" pandas numpy matplotlib


In [ ]:
# --- Minimal PlantGuard‑Meta client (self-contained) ---
from __future__ import annotations

import asyncio
from dataclasses import dataclass, asdict
from typing import Any, Dict, Literal, Optional

from openenv.core import EnvClient
from openenv.core.client_types import StepResult
from openenv.core.env_server.types import Action, Observation, State
from pydantic import Field


class PlantAction(Action):
    tool: Literal[
        "run_diagnostic",
        "adjust_load",
        "dispatch_repair",
        "order_spare_part",
        "do_nothing",
    ] = Field(...)
    reasoning: str = Field(default="")
    load_reduction: Optional[float] = Field(default=None, ge=0.1, le=0.9)


class PlantObservation(Observation):
    air_temp: float = 0.0
    process_temp: float = 0.0
    rotational_speed: float = 0.0
    torque: float = 0.0
    tool_wear: float = 0.0
    shift_hour: int = 0
    remaining_budget: float = 10000.0


class PlantState(State):
    shift_complete: bool = False
    cascade_occurred: bool = False
    budget_remaining: float = 10000.0
    spare_available: bool = False


class PlantGovernorClient(EnvClient[PlantAction, PlantObservation, PlantState]):
    def _step_payload(self, action: PlantAction) -> Dict[str, Any]:
        payload: Dict[str, Any] = {"tool": action.tool, "reasoning": action.reasoning}
        if action.load_reduction is not None:
            payload["load_reduction"] = action.load_reduction
        return payload

    def _parse_result(self, payload: Dict[str, Any]) -> StepResult[PlantObservation]:
        obs_data = payload.get("observation", {})
        observation = PlantObservation(
            air_temp=obs_data.get("air_temp", 0.0),
            process_temp=obs_data.get("process_temp", 0.0),
            rotational_speed=obs_data.get("rotational_speed", 0.0),
            torque=obs_data.get("torque", 0.0),
            tool_wear=obs_data.get("tool_wear", 0.0),
            shift_hour=obs_data.get("shift_hour", 0),
            remaining_budget=obs_data.get("remaining_budget", 0.0),
            done=payload.get("done", False),
            reward=payload.get("reward"),
            metadata=obs_data.get("metadata", {}),
        )
        return StepResult(
            observation=observation,
            reward=payload.get("reward"),
            done=payload.get("done", False),
        )

    def _parse_state(self, payload: Dict[str, Any]) -> PlantState:
        return PlantState(
            episode_id=payload.get("episode_id"),
            step_count=payload.get("step_count", 0),
            shift_complete=payload.get("shift_complete", False),
            cascade_occurred=payload.get("cascade_occurred", False),
            budget_remaining=payload.get("budget_remaining", 10000.0),
            spare_available=payload.get("spare_available", False),
        )


BASE_URL = "https://narendra78-plant-governor-env.hf.space"
print('BASE_URL =', BASE_URL)


In [ ]:
# --- Rollout episodes against the deployed environment (Kaggle-safe) ---
import csv, time


def random_action(step: int) -> PlantAction:
    import random

    tool = random.choice([
        "run_diagnostic",
        "adjust_load",
        "dispatch_repair",
        "order_spare_part",
        "do_nothing",
    ])
    lr = random.choice([None, 0.2, 0.4, 0.6, 0.8]) if tool == "adjust_load" else None
    return PlantAction(tool=tool, reasoning="random baseline", load_reduction=lr)


@dataclass
class EpisodeRow:
    episode: int
    seed: int
    steps: int
    total_reward: float
    avg_reward: float
    cascade: bool
    complete: bool
    budget_remaining: float
    spare_available: bool
    wall_time_s: float


async def run_episode(seed: int, episode_idx: int, max_steps: int = 200) -> EpisodeRow:
    t0 = time.time()
    async with PlantGovernorClient(base_url=BASE_URL) as client:
        result = await client.reset(seed=seed)
        total_reward = 0.0
        step = 0

        while not result.done and step < max_steps:
            action = random_action(step)
            result = await client.step(action)
            total_reward += float(result.reward or 0.0)
            step += 1

        st = await client.state()
        wall = time.time() - t0
        return EpisodeRow(
            episode=episode_idx,
            seed=seed,
            steps=int(st.step_count),
            total_reward=float(total_reward),
            avg_reward=float(total_reward / max(step, 1)),
            cascade=bool(st.cascade_occurred),
            complete=bool(st.shift_complete),
            budget_remaining=float(st.budget_remaining),
            spare_available=bool(st.spare_available),
            wall_time_s=float(wall),
        )


async def run_rollouts(episodes: int, max_steps: int, out_csv: str = "episode_log.csv"):
    rows = []
    for ep in range(episodes):
        seed = 1000 + ep
        r = await run_episode(seed=seed, episode_idx=ep, max_steps=max_steps)
        rows.append(r)
        print(
            f"ep={r.episode} seed={r.seed} steps={r.steps} "
            f"reward={r.total_reward:+.1f} cascade={r.cascade} complete={r.complete}"
        )

    with open(out_csv, "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=list(asdict(rows[0]).keys()))
        w.writeheader()
        for r in rows:
            w.writerow(asdict(r))

    print("\nWrote", out_csv)


# Kaggle-friendly settings
EPISODES = 20      # bump to 100+ for judging
MAX_STEPS = 200    # set 720 for full-shift rollouts
OUT_CSV = "episode_log.csv"

# Kaggle notebooks may not support top-level `await` reliably.
asyncio.run(run_rollouts(EPISODES, MAX_STEPS, OUT_CSV))


In [ ]:
# --- Plot reward curve + rolling success rate ---
import pandas as pd
import matplotlib.pyplot as plt


df = pd.read_csv("episode_log.csv").sort_values("episode")

# Reward curve
plt.figure(figsize=(8, 4))
plt.plot(df["episode"], df["total_reward"], alpha=0.35, label="episode")
plt.plot(df["episode"], df["total_reward"].rolling(10, min_periods=2).mean(), label="moving avg (w=10)")
plt.xlabel("episode")
plt.ylabel("total reward")
plt.title("Reward curve (env reward)")
plt.legend()
plt.tight_layout()
plt.savefig("reward_curve.png", dpi=180)
plt.show()

# Success rate (rolling)
plt.figure(figsize=(8, 4))
complete = df["complete"].astype(int)
plt.plot(df["episode"], complete.rolling(10, min_periods=2).mean())
plt.xlabel("episode")
plt.ylabel("rolling success rate")
plt.title("Shift completion rate (rolling)")
plt.tight_layout()
plt.savefig("success_rate.png", dpi=180)
plt.show()

print("Saved: reward_curve.png, success_rate.png")
